# 실습 12. 포아송 회귀와 음이항 회귀

사회약학 연구방법 노트의 '실습 12. 포아송 회귀와 음이항 회귀'에 딸린 노트북입니다. R survival 패키지의 cgd 자료(만성 육아종병 환자 128명의 감마 인터페론 무작위배정 시험)를 환자 단위로 요약하고, 군별 발생률과 발생률비, 오프셋을 넣은 포아송 회귀, 과산포 확인과 준포아송·강건 표준오차, 음이항 회귀를 차례로 해 봅니다. 셀을 위에서부터 차례로 실행하세요. 설명과 출력 읽는 법은 사이트의 실습 12에 있습니다.


## a. 실습 데이터 준비

이 실습이 쓰는 패키지를 불러와 버전을 확인한 뒤 cgd 자료를 읽습니다. 자료는 인터넷에서 읽어 오므로 인터넷 연결이 필요합니다.

### 셀 1. 패키지 불러오기와 버전 확인

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy, statsmodels, matplotlib

print("pandas      ", pd.__version__)
print("numpy       ", np.__version__)
print("scipy       ", scipy.__version__)
print("statsmodels ", statsmodels.__version__)
print("matplotlib  ", matplotlib.__version__)

### 셀 2. cgd 불러오기 (계수과정 형식)

In [ ]:
url = ("https://vincentarelbundock.github.io/Rdatasets/csv/"
       "survival/cgd.csv")
cgd = pd.read_csv(url).drop(columns="rownames")
print(cgd.shape, cgd["id"].nunique())
cols = ["id", "treat", "tstart", "tstop", "status", "enum"]
cgd[cols].head(12)

### 셀 3. 코딩 확인

In [ ]:
first = cgd.groupby("id").first()      # 환자마다 첫 행 하나
print(first["treat"].value_counts())
print(cgd["status"].value_counts())

### 셀 4. 환자 한 명 = 한 행으로 요약

In [ ]:
pt = cgd.groupby("id").agg(
    treat=("treat", "first"),
    age=("age", "first"),
    inherit=("inherit", "first"),
    propylac=("propylac", "first"),
    n_inf=("status", "sum"),       # 감염 건수
    days=("tstop", "max"),         # 추적 일수
).reset_index()
pt["py"] = pt["days"] / 365.25     # 인년
pt["treat"] = pd.Categorical(pt["treat"],
                             categories=["placebo", "rIFN-g"])
print(pt.shape)
pt.head()

### 셀 5. 요약이 맞는지 확인

In [ ]:
span = (cgd["tstop"] - cgd["tstart"]).groupby(cgd["id"]).sum()
print((span.values == pt["days"].values).all())  # 빈틈 없는 추적?
print(pt["n_inf"].sum(), round(pt["py"].sum(), 1))
pt["n_inf"].value_counts().sort_index()

## b. 발생률과 발생률비

### 셀 6. 군별 발생률과 정확 신뢰구간

In [ ]:
from scipy import stats

rt = pt.groupby("treat", observed=True).agg(
    patients=("id", "size"),
    events=("n_inf", "sum"),
    py=("py", "sum"))
d, T = rt["events"], rt["py"]
rt["rate"] = d / T * 100                          # 100인년당
rt["lower"] = stats.chi2.ppf(0.025, 2 * d) / 2 / T * 100
rt["upper"] = stats.chi2.ppf(0.975, 2 * d + 2) / 2 / T * 100
rt.round(2)

### 셀 7. 조발생률비와 Wald 신뢰구간

In [ ]:
irr = rt.loc["rIFN-g", "rate"] / rt.loc["placebo", "rate"]
se = np.sqrt(1 / d["rIFN-g"] + 1 / d["placebo"])  # log IRR의 SE
lo = np.exp(np.log(irr) - 1.96 * se)
hi = np.exp(np.log(irr) + 1.96 * se)
print(f"IRR = {irr:.3f}  95% CI {lo:.3f} to {hi:.3f}")
print(f"SE(log IRR) = {se:.3f}")

### 셀 8. 조건부 정확검정

In [ ]:
share = T["rIFN-g"] / T.sum()        # rIFN-g군 인년의 비율
res = stats.binomtest(int(d["rIFN-g"]), int(d.sum()), share)
print(f"expected share {share:.3f}, observed {d['rIFN-g']}/{d.sum()}")
print("exact P =", res.pvalue)
ci = res.proportion_ci()             # 정확(Clopper-Pearson) 구간
k = T["placebo"] / T["rIFN-g"]
print(f"exact 95% CI {ci.low / (1 - ci.low) * k:.3f}"
      f" to {ci.high / (1 - ci.high) * k:.3f}")

## c. 포아송 회귀와 과산포

### 셀 9. 포아송 회귀 (오프셋 = log 인년)

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

pt["inherit"] = pd.Categorical(pt["inherit"],
                               categories=["X-linked", "autosomal"])
form = "n_inf ~ treat + age + inherit + propylac"
pois = smf.glm(form, data=pt, family=sm.families.Poisson(),
               offset=np.log(pt["py"])).fit()
print(pois.summary())

### 셀 10. 발생률비와 95% 신뢰구간

In [ ]:
ci = pois.conf_int()
pd.DataFrame({"IRR": np.exp(pois.params),
              "2.5%": np.exp(ci[0]),
              "97.5%": np.exp(ci[1])}).round(3)

### 셀 11. 과산포 확인

In [ ]:
print("deviance / df:", round(pois.deviance / pois.df_resid, 3))
print("Pearson chi2 / df:",
      round(pois.pearson_chi2 / pois.df_resid, 3))

### 셀 12. 관측 빈도와 포아송 기대 빈도

In [ ]:
mu = pois.fittedvalues            # 환자별 기대 건수 (인년 반영)
k = np.arange(0, 8)
obs = [(pt["n_inf"] == i).sum() for i in k]
exp_p = [stats.poisson.pmf(i, mu).sum() for i in k]
print("zeros: observed", obs[0], " Poisson", round(exp_p[0], 1))

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(k - 0.2, obs, width=0.4, label="Observed")
ax.bar(k + 0.2, exp_p, width=0.4, label="Expected, Poisson")
ax.set_xlabel("Number of serious infections per patient")
ax.set_ylabel("Number of patients")
ax.legend()
plt.show()

### 셀 13. 강건(샌드위치) 표준오차

In [ ]:
pois_r = smf.glm(form, data=pt, family=sm.families.Poisson(),
                 offset=np.log(pt["py"])).fit(cov_type="HC0")
t = "treat[T.rIFN-g]"
print("model-based SE:", round(pois.bse[t], 3))
print("robust SE     :", round(pois_r.bse[t], 3))
print("robust 95% CI :", np.exp(pois_r.conf_int().loc[t]).round(3).values)

### 셀 14. 준포아송: 표준오차에 √φ 곱하기

In [ ]:
pois_q = smf.glm(form, data=pt, family=sm.families.Poisson(),
                 offset=np.log(pt["py"])).fit(scale="X2")
phi = pois.pearson_chi2 / pois.df_resid      # Pearson chi2 / df
print("phi =", round(phi, 3), "  sqrt(phi) =", round(np.sqrt(phi), 3))
print("same coefficients:", np.allclose(pois_q.params, pois.params))
print("quasi-Poisson SE:", round(pois_q.bse[t], 4),
      "  model-based SE x sqrt(phi):", round(pois.bse[t] * np.sqrt(phi), 4))
print("quasi-Poisson 95% CI:", np.exp(pois_q.conf_int().loc[t]).round(3).values)

## d. 음이항 회귀

### 셀 15. 음이항 회귀 (NB2)

In [ ]:
nb2 = smf.negativebinomial(form, data=pt,
                           offset=np.log(pt["py"])).fit()
print(nb2.summary())

### 셀 16. 네 방법 비교, AIC와 우도비 검정

In [ ]:
fits = {"Poisson": pois, "Quasi-Poisson": pois_q,
        "Poisson, robust SE": pois_r, "Negative binomial": nb2}
tab = pd.DataFrame(
    {"IRR": [np.exp(f.params[t]) for f in fits.values()],
     "lower": [np.exp(f.conf_int().loc[t, 0]) for f in fits.values()],
     "upper": [np.exp(f.conf_int().loc[t, 1]) for f in fits.values()],
     "SE": [f.bse[t] for f in fits.values()]},
    index=fits.keys())
print("AIC  Poisson:", round(pois.aic, 1), " NB:", round(nb2.aic, 1))
lr = 2 * (nb2.llf - pois.llf)
print("LR =", round(lr, 2), " p =", round(stats.chi2.sf(lr, 1) / 2, 4))
tab.round(3)

### 셀 17. 0건 환자 수 예측 비교

In [ ]:
a = nb2.params["alpha"]
mu_nb = nb2.predict()                 # 환자별 기대 건수
p0_nb = (1 + a * mu_nb) ** (-1 / a)   # 음이항 분포의 P(0건)
print("zeros: observed", obs[0], " Poisson", round(exp_p[0], 1),
      " NB", round(p0_nb.sum(), 1))

### 셀 18. 표준화로 발생률과 발생률차 구하기

In [ ]:
one_py = np.zeros(len(pt))            # 오프셋 ln(1인년) = 0
r0 = nb2.predict(pt.assign(treat="placebo"), offset=one_py)
r1 = nb2.predict(pt.assign(treat="rIFN-g"), offset=one_py)
print(f"standardized rate per 100 PY: placebo {100 * r0.mean():.1f},"
      f" rIFN-g {100 * r1.mean():.1f}")
print(f"ratio {r1.mean() / r0.mean():.3f},"
      f" difference {100 * (r1.mean() - r0.mean()):.1f} per 100 PY")

### 셀 19. GLM의 음이항 family와 비교

In [ ]:
nb_glm = smf.glm(form, data=pt,
                 family=sm.families.NegativeBinomial(alpha=a),
                 offset=np.log(pt["py"])).fit()
both = pd.concat({"discrete NB": nb2.params,
                  "GLM NB (alpha fixed)": nb_glm.params}, axis=1)
print(both.round(4))

### 셀 20. alpha를 빠뜨리면

In [ ]:
fam = sm.families.NegativeBinomial()   # alpha를 주지 않으면?

## 과제 정답

**과제 1.** 셀 10의 발생률비 표에서 propylac 행(예방적 항생제 사용)을 한 문장으로 옮기고, 이 숫자가 말해 주지 않는 것을 두 가지 이상 쓰세요. 예방적 항생제를 쓰지 않은 환자가 몇 명인지, 셀 15의 음이항 회귀에서는 이 행이 어떻게 나오는지도 확인하세요.

### 셀 21. 과제 1 정답. 예방적 항생제의 발생률비 읽기

In [ ]:
print(pd.crosstab(pt["propylac"], pt["treat"], margins=True))
print(pt.groupby("propylac")[["n_inf", "py"]].sum().round(1))
for name, f in [("Poisson", pois), ("NB", nb2)]:
    lo, hi = np.exp(f.conf_int().loc["propylac"])
    print(f"{name:8} IRR {np.exp(f.params['propylac']):.3f}"
          f" (95% CI {lo:.3f} to {hi:.3f}), P = {f.pvalues['propylac']:.3f}")

**과제 2.** 셀 9의 포아송 회귀에서 오프셋을 (가) 인년 대신 일수의 로그 `np.log(pt["days"])`로 바꾸고 (나) 아예 빼고 다시 적합하세요. 치료군 발생률비와 절편이 각각 어떻게 달라지는지, 왜 그런지 설명하세요.

### 셀 22. 과제 2 정답. 오프셋의 단위를 바꾸거나 빼기

In [ ]:
m_days = smf.glm(form, data=pt, family=sm.families.Poisson(),
                 offset=np.log(pt["days"])).fit()     # 오프셋을 일 단위로
m_none = smf.glm(form, data=pt,
                 family=sm.families.Poisson()).fit()  # 오프셋 없음
coefs = pd.DataFrame({"ln(PY)": pois.params, "ln(days)": m_days.params,
                      "none": m_none.params})
print(coefs.round(4))
print("ln(365.25) =", round(np.log(365.25), 4))
print("treat IRR:", np.exp(coefs.loc[t]).round(3).tolist())
pt.groupby("treat", observed=True)["days"].mean().round(1)

**과제 3.** 동료가 환자 단위로 요약하지 않고 계수과정 형식의 203행을 그대로 분석했습니다. 행마다 status를 결과로, 그 구간 길이(인년)의 로그를 오프셋으로 넣은 포아송 회귀에 강건 표준오차(HC0)를 쓰고, "강건 표준오차를 썼으므로 과산포도 처리했다. 보정 발생률비는 0.36(95% CI 0.22–0.60)으로 환자 단위 분석보다 정밀하다"라고 썼습니다. 동료의 분석을 재현하고, 맞는 곳과 틀린 곳을 찾아 설명하세요.

### 셀 23. 과제 3 정답. 203행 자료에 강건 표준오차를 쓴 분석

In [ ]:
rows = cgd.copy()                                        # 동료가 쓴 203행 자료
rows["py_row"] = (rows["tstop"] - rows["tstart"]) / 365.25
rows["treat"] = pd.Categorical(rows["treat"],
                               categories=["placebo", "rIFN-g"])
rows["inherit"] = pd.Categorical(rows["inherit"],
                                 categories=["X-linked", "autosomal"])
f_row = "status ~ treat + age + inherit + propylac"
glm_row = smf.glm(f_row, data=rows, family=sm.families.Poisson(),
                  offset=np.log(rows["py_row"]))
m_hc0 = glm_row.fit(cov_type="HC0")                     # 동료의 분석
m_cl = glm_row.fit(cov_type="cluster", cov_kwds={"groups": rows["id"]})
fs = {"patients, model-based": pois, "patients, robust": pois_r,
      "intervals, robust (HC0)": m_hc0,
      "intervals, clustered by id": m_cl}
pd.DataFrame(
    {"coef": [f.params[t] for f in fs.values()],
     "SE": [f.bse[t] for f in fs.values()],
     "lower": [np.exp(f.conf_int().loc[t, 0]) for f in fs.values()],
     "upper": [np.exp(f.conf_int().loc[t, 1]) for f in fs.values()]},
    index=fs.keys()).round(4)